In [15]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv

In [18]:
load_dotenv()

chat_mistral = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="text-generation",
    temperature=1.2,
)

model = ChatHuggingFace(llm=chat_mistral)

In [3]:
documents = [

    # 🔥 FastAPI (different phrasings)
    {"id": 1, "text": "FastAPI is a modern Python framework used to build APIs quickly and efficiently.", "metadata": {"topic": "fastapi"}},
    
    {"id": 2, "text": "FastAPI supports asynchronous programming and is known for high performance.", "metadata": {"topic": "fastapi"}},
    
    {"id": 3, "text": "FastAPI automatically generates API documentation using OpenAPI and Swagger UI.", "metadata": {"topic": "fastapi"}},

    {"id": 4, "text": "Developers prefer FastAPI because it is faster than Flask and easier to use than Django in many cases.", "metadata": {"topic": "fastapi"}},


    # ⚡ Hidden related info (important for multi-query)
    {"id": 5, "text": "Async programming allows handling multiple requests concurrently, improving API performance.", "metadata": {"topic": "async"}},
    
    {"id": 6, "text": "Swagger UI provides interactive API documentation for testing endpoints easily.", "metadata": {"topic": "docs"}},


    # 🚀 Other frameworks (confusion layer)
    {"id": 7, "text": "Flask is a lightweight Python framework used for building simple web applications.", "metadata": {"topic": "flask"}},
    
    {"id": 8, "text": "Django is a full-stack Python framework that includes ORM and authentication features.", "metadata": {"topic": "django"}},


    # 🧠 Vector DB (different domain noise)
    {"id": 9, "text": "Vector databases are used to store embeddings and perform similarity search.", "metadata": {"topic": "vector_db"}},
    
    {"id": 10, "text": "Embeddings convert text into vectors to capture semantic meaning.", "metadata": {"topic": "embeddings"}},

]

In [28]:
query = "How do i improve the performance of my backend API?"

In [5]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"}
)

In [6]:
vector_store = FAISS.from_documents(
    documents = [Document(page_content=doc["text"], metadata=doc["metadata"]) for doc in documents],
    embedding = embedding
)

In [12]:
docs_dict = vector_store.docstore._dict

for doc_id, doc in docs_dict.items():
    print(doc_id, doc.page_content)

cfcd1a40-5eb3-4452-bda5-ad774bc925cc FastAPI is a modern Python framework used to build APIs quickly and efficiently.
5fbd1403-6841-4eb3-bed7-db84d604eb08 FastAPI supports asynchronous programming and is known for high performance.
4a398bac-853a-44b8-ac26-12bfa9d2f841 FastAPI automatically generates API documentation using OpenAPI and Swagger UI.
0a44a159-c132-47e7-af6c-fdccfc62df16 Developers prefer FastAPI because it is faster than Flask and easier to use than Django in many cases.
d9570ec3-6ea7-4da4-9e1b-a529299e08f3 Async programming allows handling multiple requests concurrently, improving API performance.
23084b03-9265-4300-8772-7c660acf21f1 Swagger UI provides interactive API documentation for testing endpoints easily.
a9e50b9d-bfc5-43eb-816a-f765d5d0b7dc Flask is a lightweight Python framework used for building simple web applications.
09331982-c6d8-427e-84a0-9b09e0840c72 Django is a full-stack Python framework that includes ORM and authentication features.
17d2722a-9c3b-4afc-b

In [13]:
similarity_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [25]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5}),
    llm = model                 
)

In [29]:
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [30]:
for result in similarity_results:
    print("Similarity Result:", result.page_content)

print("\n---\n")

for result in multiquery_results:
    print("Multi-Query Result:", result.page_content)

Similarity Result: Async programming allows handling multiple requests concurrently, improving API performance.
Similarity Result: FastAPI is a modern Python framework used to build APIs quickly and efficiently.
Similarity Result: Developers prefer FastAPI because it is faster than Flask and easier to use than Django in many cases.
Similarity Result: FastAPI supports asynchronous programming and is known for high performance.

---

Multi-Query Result: Async programming allows handling multiple requests concurrently, improving API performance.
Multi-Query Result: FastAPI supports asynchronous programming and is known for high performance.
Multi-Query Result: FastAPI is a modern Python framework used to build APIs quickly and efficiently.
Multi-Query Result: FastAPI automatically generates API documentation using OpenAPI and Swagger UI.
Multi-Query Result: Developers prefer FastAPI because it is faster than Flask and easier to use than Django in many cases.
Multi-Query Result: Swagger UI